# 04 — Augmentación con LLM (Claude API)

Generamos ejemplos sintéticos en español del siglo XVI-XVII para las clases con pocos datos de entrenamiento:
- **surprise** (sorpresa): 9 ejemplos reales → +40 sintéticos
- **anger** (ira): 12 ejemplos reales → +40 sintéticos

El resultado se guarda en `train/train_augmented.csv` para re-entrenar el modelo.

In [1]:
import os
import json
import time
import pandas as pd
import ollama

EMOTION_COLS = ['anger', 'fear', 'joy', 'sadness', 'surprise', 'hope']
TRAIN_PATH   = '../train/train.csv'
OUT_PATH     = '../train/train_augmented.csv'
MODEL        = 'llama3.1:8b'

train_df = pd.read_csv(TRAIN_PATH).dropna(subset=['text'])
for col in EMOTION_COLS:
    train_df[col] = train_df[col].fillna(0).astype(int)

print(f'Train original: {len(train_df)} filas')
print(train_df[EMOTION_COLS].sum().sort_values())

Train original: 2657 filas
surprise       9
anger         12
hope          73
joy          129
fear         303
sadness     1180
dtype: int64


## Configuración — Ollama (local, gratuito)

Usamos `llama3.1:8b` corriendo localmente via Ollama. No se necesita API key ni conexión a internet.  
Asegúrate de que Ollama esté arrancado (`ollama serve`) antes de ejecutar.

In [2]:
# Verifica que Ollama responde
try:
    r = ollama.chat(model=MODEL, messages=[{'role': 'user', 'content': 'Responde solo: OK'}])
    print(f'Ollama [{MODEL}] listo:', r['message']['content'].strip())
except Exception as e:
    print(f'ERROR: {e}')
    print('Asegúrate de que Ollama está arrancado: ejecuta `ollama serve` en otro terminal.')

Ollama [llama3.1:8b] listo: ¿En qué puedo ayudarte?


In [3]:
def get_real_examples(df: pd.DataFrame, emotion: str, n: int = 5) -> list[str]:
    """Devuelve hasta n fragmentos reales que tienen la emoción activa."""
    subset = df[df[emotion] == 1]['text'].dropna().tolist()
    return subset[:n]


# Muestra los ejemplos reales de surprise y anger
for emo in ['surprise', 'anger']:
    examples = get_real_examples(train_df, emo)
    print(f'\n--- {emo.upper()} ({len(train_df[train_df[emo]==1])} ejemplos reales) ---')
    for i, ex in enumerate(examples, 1):
        print(f'  {i}. {ex[:120]}')


--- SURPRISE (9 ejemplos reales) ---
  1. 0
  2. 1
  3. Todo me da mayor deseo de preguntarte el cómo y en qué y de ver a mi hija con quien sin duda alguna creo estarás content
  4. Y luego otro día martes acabando de comulgar oí en mi interior que me fuese luego porque había de ver grandes cosas.
  5. Y luego sentí como se abrazó con mi espíritu el arzobispo Bartolomé exhortándome a grandes cosas.

--- ANGER (12 ejemplos reales) ---
  1. Sabed que Benavente está muy enojado de vos porque habéis jurado contra él.
  2. Es menester brava prisa porque de cada palabrita quieren esa mala gente que se dispute todo al fin de largas por acabarm
  3. Por un solo Dios que con todas veras se eche el resto a esto aunque más esa mala gente lo estorbe con sus enredos y mara
  4. Y si no mándeme perdonar porque quiero cobrar lo mío y porque creo que no me querrá por deservidor.
  5. 1


In [4]:
EMOTION_DESCRIPTIONS = {
    'surprise': 'sorpresa o asombro (algo inesperado, una noticia imprevista, una situación extraordinaria)',
    'anger':    'ira, enojo o indignación (queja fuerte, reproche, frustración intensa)',
    'joy':      'alegría, satisfacción o alivio',
    'fear':     'miedo, temor o angustia ante una amenaza o peligro',
    'sadness':  'tristeza, pena, dolor emocional o duelo',
    'hope':     'esperanza, deseo de que algo bueno ocurra en el futuro',
}


def build_prompt(emotion: str, real_examples: list[str], n_generate: int = 10) -> str:
    emotion_desc = EMOTION_DESCRIPTIONS[emotion]
    examples_text = '\n'.join(f'  - "{ex}"' for ex in real_examples)

    return f"""Eres un experto en literatura epistolar española de los siglos XVI y XVII.
Tu tarea es generar fragmentos sintéticos de cartas históricas en castellano antiguo.

EMOCIÓN OBJETIVO: {emotion.upper()} — {emotion_desc}.

EJEMPLOS REALES del corpus (para calibrar el estilo y la emoción):
{examples_text}

INSTRUCCIONES:
- Genera exactamente {n_generate} fragmentos nuevos y distintos.
- Cada fragmento debe expresar claramente la emoción de {emotion.upper()}.
- Usa lengua castellana del siglo XVI-XVII: formas como "VM" (vuestra merced), "ansí", "deste",
  "della", "quel", "holgar", "merced", verbos en segunda persona de cortesía, etc.
- Los fragmentos deben tener entre 15 y 80 palabras.
- NO incluyas otras emociones fuertes que compitan con {emotion.upper()}.
- Devuelve ÚNICAMENTE un objeto JSON con la clave "fragments": una lista de {n_generate} strings.
  Sin explicaciones adicionales.

Ejemplo de formato esperado:
{{"fragments": ["Fragmento 1...", "Fragmento 2...", ...]}}"""

In [5]:
import re

def extract_fragments(raw: str, n_expected: int) -> list[str]:
    """Intenta extraer fragmentos del texto aunque el JSON no sea perfecto."""
    # Intento 1: JSON estándar
    start, end = raw.find('{'), raw.rfind('}') + 1
    if start != -1 and end > 0:
        try:
            data = json.loads(raw[start:end])
            frags = data.get('fragments', [])
            if frags:
                return frags
        except json.JSONDecodeError:
            pass

    # Intento 2: buscar lista JSON directamente ["...", "..."]
    match = re.search(r'\[.*?\]', raw, re.DOTALL)
    if match:
        try:
            frags = json.loads(match.group())
            if isinstance(frags, list) and frags:
                return frags
        except json.JSONDecodeError:
            pass

    # Intento 3: extraer líneas numeradas o con guion
    lines = [l.strip().lstrip('0123456789.-)"\'• ') for l in raw.split('\n')]
    frags = [l for l in lines if len(l) > 20]
    return frags[:n_expected]


def generate_synthetic(emotion: str, n_total: int = 40, batch_size: int = 10,
                        max_retries: int = 3) -> list[str]:
    real_examples = get_real_examples(train_df, emotion, n=5)
    all_fragments = []
    n_batches = n_total // batch_size

    for i in range(n_batches):
        print(f'  Batch {i+1}/{n_batches}...', end=' ', flush=True)
        fragments = []

        for attempt in range(max_retries):
            prompt = build_prompt(emotion, real_examples, n_generate=batch_size)
            response = ollama.chat(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                options={'temperature': 0.85},
            )
            raw = response['message']['content'].strip()
            fragments = extract_fragments(raw, batch_size)
            if len(fragments) >= batch_size // 2:  # acepta si tiene al menos la mitad
                break
            print(f'reintento {attempt+1}...', end=' ', flush=True)

        if fragments:
            all_fragments.extend(fragments[:batch_size])
            print(f'OK ({len(fragments[:batch_size])} fragmentos)')
        else:
            print('WARN: sin fragmentos tras reintentos')

    print(f'  Total generados para [{emotion}]: {len(all_fragments)}')
    return all_fragments

In [7]:
# Carga el train_augmented existente (ya tiene surprise+anger)
# para añadir fear, joy y hope sin duplicar
import os

BASE_PATH = OUT_PATH if os.path.exists(OUT_PATH) else TRAIN_PATH
base_df   = pd.read_csv(BASE_PATH).dropna(subset=['text'])
for col in EMOTION_COLS:
    base_df[col] = base_df[col].fillna(0).astype(int)

print(f'Base para augmentación: {len(base_df)} filas')
print(base_df[EMOTION_COLS].sum().sort_values())

# Emociones nuevas a aumentar (surprise y anger ya están)
AUGMENT_CONFIG = {
    'fear': 40,
    'joy':  40,
    'hope': 40,
}

synthetic_rows = []

for emotion, n_total in AUGMENT_CONFIG.items():
    print(f'\nGenerando {n_total} ejemplos para [{emotion}]...')
    fragments = generate_synthetic(emotion, n_total=n_total, batch_size=10)
    for text in fragments:
        row = {'text': text}
        for col in EMOTION_COLS:
            row[col] = 1 if col == emotion else 0
        synthetic_rows.append(row)

synth_df = pd.DataFrame(synthetic_rows)
print(f'\nTotal filas sintéticas nuevas: {len(synth_df)}')
print(synth_df[EMOTION_COLS].sum())

Base para augmentación: 2737 filas
surprise      49
anger         52
hope          73
joy          129
fear         303
sadness     1180
dtype: int64

Generando 40 ejemplos para [fear]...
  Batch 1/4... OK (10 fragmentos)
  Batch 2/4... OK (10 fragmentos)
  Batch 3/4... reintento 1... OK (10 fragmentos)
  Batch 4/4... OK (10 fragmentos)
  Total generados para [fear]: 40

Generando 40 ejemplos para [joy]...
  Batch 1/4... OK (10 fragmentos)
  Batch 2/4... reintento 1... OK (10 fragmentos)
  Batch 3/4... OK (10 fragmentos)
  Batch 4/4... reintento 1... OK (10 fragmentos)
  Total generados para [joy]: 40

Generando 40 ejemplos para [hope]...
  Batch 1/4... OK (10 fragmentos)
  Batch 2/4... reintento 1... reintento 2... OK (10 fragmentos)
  Batch 3/4... reintento 1... OK (10 fragmentos)
  Batch 4/4... reintento 1... reintento 2... reintento 3... OK (1 fragmentos)
  Total generados para [hope]: 31

Total filas sintéticas nuevas: 111
anger        0
fear        40
joy         40
sadness      

In [8]:
# Inspecciona algunos ejemplos generados antes de guardar
for emotion in AUGMENT_CONFIG:
    print(f'\n--- EJEMPLOS SINTÉTICOS [{emotion.upper()}] ---')
    samples = synth_df[synth_df[emotion] == 1]['text'].head(3).tolist()
    for s in samples:
        print(f'  · {s}')


--- EJEMPLOS SINTÉTICOS [FEAR] ---
  · Aquí te presento los fragmentos sintéticos de cartas históricas en castellano antiguo que expresan la emoción de FEAR:
  · Me aflige el pensamiento de no poderme defender contra tan malvados enemigos, VM.",
  · Ansí me temo por mi honor y mi vida si no puedo salir de este trance difícil.",

--- EJEMPLOS SINTÉTICOS [JOY] ---
  · Vea Vuestra Excelencia que su carta me ha llegado en buen día y me ha dado tanta alegría que holgo mucho
  · Mi corazón se llena de gozo cuando pienso en la felicidad de mi señora, Dios la guarde
  · Hoy he recibido la noticia de su recuperación y es un consuelo infinito para mí

--- EJEMPLOS SINTÉTICOS [HOPE] ---
  · Ansí espero y deseo a Dios sea servido de satisfacer mi voluntad, merced.
  · VM no se ofenda conmigo si escribo desta manera, pues ansí es el estado en que me hallo.
  · Dios me dé buena suerte, holgaré cuando esté en la corte y pueda verme libre de estos trabajos.


In [9]:
# Combina base (train original + surprise/anger) + nuevos sintéticos
augmented_df = pd.concat([base_df, synth_df], ignore_index=True)
augmented_df.to_csv(OUT_PATH, index=False)

print(f'Base            : {len(base_df)} filas')
print(f'Nuevos sintéticos: {len(synth_df)} filas')
print(f'Train aumentado : {len(augmented_df)} filas  →  guardado en {OUT_PATH}')
print()
print('Distribución final:')
print(augmented_df[EMOTION_COLS].sum().sort_values(ascending=False))

Base            : 2737 filas
Nuevos sintéticos: 111 filas
Train aumentado : 2848 filas  →  guardado en ../train/train_augmented.csv

Distribución final:
sadness     1180
fear         343
joy          169
hope         104
anger         52
surprise      49
dtype: int64


## Siguiente paso

Una vez generado `train/train_augmented.csv`, abre el **notebook 03** y cambia:

```python
CFG['train_path'] = '../train/train_augmented.csv'
CFG['num_epochs'] = 3
```

Y re-entrena. El modelo ahora tendrá 40+ ejemplos de `surprise` y `anger` en lugar de 9 y 12.